# Pub/Sub → BigQuery 연동 실습 : append 와 upsert 비교

**목표** : 같은 Pub/Sub 토픽의 메시지를 두 경로로 받아서 차이를 확인한다.

| 경로 | 구독 종류 | 적재 방식 | 테이블 |
|---|---|---|---|
| ① append | BigQuery 구독 (콘솔/gcloud 설정, 코드 없음) | 메시지 1건 = 행 1건 추가 | `orders_stream_append` |
| ② upsert | Pull 구독 + Python Storage Write API | `_CHANGE_TYPE` = `UPSERT` / `DELETE` 로 행 갱신·삭제 | `orders_upsert` (PK 필요) |

```
gcloud pubsub topics publish orders-topic
   ├─ orders-append-sub  (BigQuery 구독) ──────────→ orders_stream_append   (append)
   └─ orders-upsert-sub     (Pull, Python 노트북) ───→ orders_upsert              (UPSERT / DELETE)
```

**실습 원칙**
- 메시지는 **한 건씩 발행 → 소비 셀 실행** 순서로 진행한다. (여러 건을 한 번에 pull 하면 처리 순서가 보장되지 않는다.)
- 결과 확인은 **콘솔과 SQL** 로 한다.
- 리전은 **asia-northeast3 (서울)** 로 통일한다.

**사전 준비**
- 노트북 실행 계정에 `BigQuery Data Editor`, `Pub/Sub Subscriber`, `Pub/Sub Viewer` 권한
- BigQuery 구독은 Pub/Sub 서비스 에이전트에 대상 테이블 쓰기 권한(`BigQuery Data Editor`)이 필요하다. ([문서 기준, 리허설에서 확인 필요])

## 1. 리소스 만들기

### 1-1. 데이터셋 (SQL)

```sql
CREATE SCHEMA IF NOT EXISTS `YOUR_PROJECT_ID.write_demo`
OPTIONS (location = 'asia-northeast3');
```

### 1-2. 테이블 (SQL)
`orders_upsert` 는 upsert 를 쓰려면 **Primary Key 선언이 필수**다. (`NOT ENFORCED` : BigQuery 가 중복을 막아주지 않으므로 키는 유일해야 한다.)

```sql
-- ① append 용 테이블 (BigQuery 구독이 채운다)
CREATE TABLE IF NOT EXISTS `YOUR_PROJECT_ID.write_demo.orders_stream_append` (
    order_id    INT64,
    user_id     INT64,
    status      STRING,
    amount      INT64,
    change_type STRING
);

-- ② upsert 용 테이블 (Storage Write API upsert 가 채운다)
CREATE TABLE IF NOT EXISTS `YOUR_PROJECT_ID.write_demo.orders_upsert` (
    order_id INT64 PRIMARY KEY NOT ENFORCED,
    user_id  INT64,
    status   STRING,
    amount   INT64
);
```

### 1-3. 토픽과 구독 (gcloud)
- `orders-append-sub` : BigQuery 구독. `--use-table-schema` 로 메시지 필드를 테이블 컬럼에 맞춰 적재한다. ([문서 기준, 리허설에서 확인 필요] 메시지 필드 중 테이블에 없는 값의 처리 방식)
- `orders-upsert-sub` : 일반 Pull 구독. 노트북의 Python 이 가져간다.

```bash
gcloud pubsub topics create orders-topic --project=YOUR_PROJECT_ID

gcloud pubsub subscriptions create orders-append-sub \
    --topic=orders-topic \
    --bigquery-table=YOUR_PROJECT_ID:write_demo.orders_stream_append \
    --use-table-schema

gcloud pubsub subscriptions create orders-upsert-sub \
    --topic=orders-topic
```

## 2. Python 준비 (Storage Write API)
upsert 행은 protobuf 로 보낸다. 아래 셀은 행 구조(`OrderRow`)와 쓰기 스트림을 만드는 준비 코드다. (protobuf 의 동적 descriptor 방식, 리허설에서 확인 필요)

In [ ]:
PROJECT = "YOUR_PROJECT_ID"
DATASET = "write_demo"
UPSERT_TABLE = "orders_upsert"
UPSERT_SUB = "orders-upsert-sub"

import json
from google.protobuf import descriptor_pb2, descriptor_pool, message_factory
from google.cloud import bigquery_storage_v1, pubsub_v1
from google.cloud.bigquery_storage_v1 import types, writer

# 행 구조 : _CHANGE_TYPE 필드가 UPSERT / DELETE 를 결정한다
F = descriptor_pb2.FieldDescriptorProto
row_proto = descriptor_pb2.DescriptorProto(name="OrderRow")
for name, num, typ in [
    ("order_id", 1, F.TYPE_INT64),
    ("user_id", 2, F.TYPE_INT64),
    ("status", 3, F.TYPE_STRING),
    ("amount", 4, F.TYPE_INT64),
    ("_CHANGE_TYPE", 5, F.TYPE_STRING),
]:
    row_proto.field.add(name=name, number=num, type=typ, label=F.LABEL_OPTIONAL)

file_proto = descriptor_pb2.FileDescriptorProto(
    name="order_row.proto", package="write_demo", message_type=[row_proto], syntax="proto2"
)
pool = descriptor_pool.DescriptorPool()
pool.Add(file_proto)
OrderRow = message_factory.GetMessageClass(pool.FindMessageTypeByName("write_demo.OrderRow"))

# 기본 스트림(_default)에 쓰기 : upsert 는 기본 스트림을 써야 한다
bq_storage = bigquery_storage_v1.BigQueryWriteClient()
write_stream = f"projects/{PROJECT}/datasets/{DATASET}/tables/{UPSERT_TABLE}/_default"
template = types.AppendRowsRequest(
    write_stream=write_stream,
    proto_rows=types.AppendRowsRequest.ProtoData(
        writer_schema=types.ProtoSchema(proto_descriptor=row_proto)
    ),
)
append_stream = writer.AppendRowsStream(bq_storage, template)

sub_client = pubsub_v1.SubscriberClient()
sub_path = sub_client.subscription_path(PROJECT, UPSERT_SUB)
print("준비 완료 :", write_stream)

## 3. 소비 : Pull → Storage Write API (UPSERT / DELETE)
아래 함수는 `orders-upsert-sub` 에서 메시지를 가져와 `_CHANGE_TYPE` 과 함께 적재하고 ack 한다. 시나리오마다 한 번씩 실행한다.

In [ ]:
def consume_upsert():
    resp = sub_client.pull(request={"subscription": sub_path, "max_messages": 10})
    if not resp.received_messages:
        print("pull 된 메시지 없음")
        return

    rows = []
    for m in resp.received_messages:
        body = json.loads(m.message.data)
        rows.append(
            OrderRow(
                order_id=body["order_id"],
                user_id=body["user_id"],
                status=body["status"],
                amount=body["amount"],
                _CHANGE_TYPE=body["change_type"],   # UPSERT 또는 DELETE
            ).SerializeToString()
        )

    request = types.AppendRowsRequest(
        proto_rows=types.AppendRowsRequest.ProtoData(rows=types.ProtoRows(serialized_rows=rows))
    )
    append_stream.send(request).result()   # 적재 완료를 기다린다

    sub_client.acknowledge(
        request={"subscription": sub_path, "ack_ids": [m.ack_id for m in resp.received_messages]}
    )
    print(f"{len(rows)}건 적재 후 ack 완료")

## 4. 시나리오
각 시나리오는 **발행(gcloud) → 소비(Python 셀) → 확인(SQL)** 순서다.

### 시나리오 1 : 주문 생성 (UPSERT)

```bash
gcloud pubsub topics publish orders-topic \
    --message='{"order_id":1,"user_id":10,"status":"PAID","amount":15000,"change_type":"UPSERT"}'
```

In [ ]:
consume_upsert()

예상 결과 : `orders_stream_append` 1행, `orders_upsert` 1행 (`PAID`)

### 시나리오 2 : 같은 주문 상태 변경 (같은 키 UPSERT)

```bash
gcloud pubsub topics publish orders-topic \
    --message='{"order_id":1,"user_id":10,"status":"SHIPPED","amount":15000,"change_type":"UPSERT"}'
```

In [ ]:
consume_upsert()

예상 결과 : `orders_stream_append` 2행 (이력이 쌓임), `orders_upsert` 는 여전히 **1행**, 상태만 `SHIPPED` 로 바뀜

### 시나리오 3 : 주문 삭제 (DELETE)

```bash
gcloud pubsub topics publish orders-topic \
    --message='{"order_id":1,"user_id":10,"status":"SHIPPED","amount":15000,"change_type":"DELETE"}'
```

In [ ]:
consume_upsert()

예상 결과 : `orders_stream_append` 3행 (DELETE 이력도 남음), `orders_upsert` **0행**. ([문서 기준] DELETE 는 적용 워터마크가 지나야 반영된다고 설명되므로, 바로 조회해서 행이 남아 있으면 잠시 뒤 다시 확인 — **리허설에서 확인 필요**)

## 5. 결과 비교 (SQL)
두 테이블을 나란히 본다.

```sql
-- ① append : 이벤트 이력 (모든 메시지가 행으로 남는다)
SELECT order_id, status, amount, change_type
FROM `YOUR_PROJECT_ID.write_demo.orders_stream_append`
ORDER BY order_id;

-- ② upsert : 현재 상태 (같은 키는 1행)
SELECT order_id, status, amount
FROM `YOUR_PROJECT_ID.write_demo.orders_upsert`
ORDER BY order_id;
```

## 6. 정리 : append vs upsert

| 구분 | ① BigQuery 구독 (append) | ② Storage Write API (upsert) |
|---|---|---|
| 코드 | 없음 (gcloud 설정) | Python (protobuf) |
| 같은 키 재전송 | 행이 추가됨 (이력) | 같은 행이 갱신됨 (상태) |
| 삭제 | 삭제 이벤트도 행으로 남음 | 행이 삭제됨 |
| 필요한 것 | 테이블 스키마 일치 | Primary Key 선언, 기본 스트림 |
| 주의 | 메시지가 중복 전달되면 행도 중복 (at-least-once) | 키 중복 방지는 BigQuery 가 하지 않음, 순서가 어긋나면 결과가 달라질 수 있음 (`_CHANGE_SEQUENCE_NUMBER` 로 순서 지정 가능, 3번 실습에서 다룸) |

> 이 노트북은 설명 중심의 최소 예제다. 실서비스에서는 메시지 재시도, 실패 처리(dead letter), 순서 보장을 함께 설계해야 한다.